<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/14-training-that-converges.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 14 — Training That Converges

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/training-that-converges/).

A deeper network in NumPy, and the four things that decide whether it trains at all: initialisation, the activation function, the optimiser, and regularisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_raw_tr, X_raw_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_raw_tr)
X_tr, X_te = scaler.transform(X_raw_tr), scaler.transform(X_raw_te)
Y_tr = np.eye(10)[y_tr]                                          # one-hot targets for softmax
print(f'{len(y_tr)} training digits, {len(y_te)} test digits, 64 features, 10 classes')

## 1. A network of any depth, in NumPy

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z): return np.maximum(0, z)

ACT = {'sigmoid': (sigmoid, lambda a: a * (1 - a)),              # derivative written in terms of the activation
       'tanh': (np.tanh, lambda a: 1 - a ** 2),
       'relu': (relu, lambda a: (a > 0).astype(float))}

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def init(sizes, scheme, seed=0):
    rng = np.random.default_rng(seed)
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        if scheme == 'zeros':      W = np.zeros((n_in, n_out))
        elif scheme == 'large':    W = rng.normal(0, 1.0, (n_in, n_out))
        elif scheme == 'small':    W = rng.normal(0, 0.01, (n_in, n_out))
        elif scheme == 'xavier':   W = rng.normal(0, np.sqrt(1 / n_in), (n_in, n_out))
        elif scheme == 'he':       W = rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out))
        params.append([W, np.zeros(n_out)])
    return params

def forward(X, params, act):
    f, _ = ACT[act]
    activations = [X]
    for i, (W, b) in enumerate(params):
        z = activations[-1] @ W + b
        activations.append(softmax(z) if i == len(params) - 1 else f(z))
    return activations

def backward(X, Y, params, act):
    _, df = ACT[act]
    acts = forward(X, params, act)
    n = len(X)
    delta = (acts[-1] - Y) / n                                   # softmax + cross-entropy: the same clean gradient
    grads = []
    for i in reversed(range(len(params))):
        W, b = params[i]
        grads.insert(0, [acts[i].T @ delta, delta.sum(axis=0)])
        if i > 0:
            delta = (delta @ W.T) * df(acts[i])
    return grads, acts

def loss(X, Y, params, act):
    p = forward(X, params, act)[-1]
    return -np.mean(np.sum(Y * np.log(p + 1e-12), axis=1))

def accuracy(X, y, params, act):
    return np.mean(forward(X, params, act)[-1].argmax(axis=1) == y)

## 2. Vanishing gradients

In [ ]:
sizes = [64, 64, 64, 64, 64, 64, 10]                             # five hidden layers
for act, scheme in [('sigmoid', 'xavier'), ('tanh', 'xavier'), ('relu', 'he')]:
    params = init(sizes, scheme, seed=0)
    grads, _ = backward(X_tr[:256], Y_tr[:256], params, act)
    norms = [np.linalg.norm(g[0]) for g in grads]
    print(f'{act:<8} gradient size by layer, first to last: ' + ' '.join(f'{n:8.5f}' for n in norms))

## 3. Initialisation

In [ ]:
def activation_spread(params, act):
    acts = forward(X_tr[:256], params, act)
    return [a.std() for a in acts[1:-1]]

for scheme in ['zeros', 'small', 'large', 'xavier', 'he']:
    params = init(sizes, scheme, seed=0)
    spread = activation_spread(params, 'relu')
    print(f'{scheme:<7} relu activation spread by hidden layer: ' + ' '.join(f'{s:7.3f}' for s in spread))

In [ ]:
# Zeros: every unit in a layer is identical and stays identical
params = init([64, 32, 10], 'zeros')
grads, _ = backward(X_tr[:256], Y_tr[:256], params, 'relu')
print('with zero weights, the gradient for every hidden unit is the same:', np.allclose(grads[0][0], grads[0][0][:, :1]))

## 4. Optimisers

In [ ]:
def train(params, act, optimiser, rate, epochs=30, batch=64, seed=0, decay=0.0, X=X_tr, Y=Y_tr, log=None):
    rng = np.random.default_rng(seed)
    state = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]      # momentum / first moment
    state2 = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]     # second moment, for RMSprop and Adam
    t = 0
    history = []
    for epoch in range(epochs):
        for start in range(0, len(X), batch):
            idx = rng.permutation(len(X))[:batch] if start == 0 else idx_order[start:start + batch]
            if start == 0:
                idx_order = rng.permutation(len(X)); idx = idx_order[:batch]
            grads, _ = backward(X[idx], Y[idx], params, act)
            t += 1
            for i, ((W, b), (gW, gb)) in enumerate(zip(params, grads)):
                gW = gW + decay * W                                           # weight decay, chapter 5's ridge penalty
                for j, (P, g) in enumerate([(W, gW), (b, gb)]):
                    if optimiser == 'sgd':
                        P -= rate * g
                    elif optimiser == 'momentum':
                        state[i][j] = 0.9 * state[i][j] + g
                        P -= rate * state[i][j]
                    elif optimiser == 'rmsprop':
                        state2[i][j] = 0.9 * state2[i][j] + 0.1 * g ** 2
                        P -= rate * g / (np.sqrt(state2[i][j]) + 1e-8)
                    elif optimiser == 'adam':
                        state[i][j] = 0.9 * state[i][j] + 0.1 * g
                        state2[i][j] = 0.999 * state2[i][j] + 0.001 * g ** 2
                        m_hat = state[i][j] / (1 - 0.9 ** t)
                        v_hat = state2[i][j] / (1 - 0.999 ** t)
                        P -= rate * m_hat / (np.sqrt(v_hat) + 1e-8)
        history.append(loss(X, Y, params, act))
        if log and (epoch + 1) in log:
            print(f'    epoch {epoch + 1:>3}  train loss {history[-1]:.4f}  test accuracy {accuracy(X_te, y_te, params, act):.3f}')
    return params, history

sizes = [64, 64, 64, 10]
print(f'{"optimiser":<10} {"rate":>6} {"loss after 5":>13} {"after 30":>9} {"test acc":>9}')
for opt, rate in [('sgd', 0.01), ('sgd', 0.1), ('momentum', 0.01), ('rmsprop', 0.001), ('adam', 0.001)]:
    params, hist = train(init(sizes, 'he', seed=0), 'relu', opt, rate)
    print(f'{opt:<10} {rate:>6} {hist[4]:>13.4f} {hist[-1]:>9.4f} {accuracy(X_te, y_te, params, "relu"):>9.3f}')

## 5. The learning rate, once more

In [ ]:
print(f'{"rate":>7} {"loss after 30":>14} {"test acc":>9}')
for rate in [0.0001, 0.001, 0.01, 0.1]:
    params, hist = train(init(sizes, 'he', seed=0), 'relu', 'adam', rate)
    final = hist[-1]
    print(f'{rate:>7} {final:>14.4f} {accuracy(X_te, y_te, params, "relu"):>9.3f}' + ('   diverged' if not np.isfinite(final) or final > 2 else ''))

## 6. Overfit one batch first

In [ ]:
tiny = slice(0, 16)
params, hist = train(init(sizes, 'he', seed=0), 'relu', 'adam', 0.001, epochs=200, batch=16, X=X_tr[tiny], Y=Y_tr[tiny])
print(f'16 digits, 200 epochs: training loss {hist[-1]:.5f}, training accuracy {accuracy(X_tr[tiny], y_tr[tiny], params, "relu"):.2f}')

## 7. Overfitting, and three ways to hold it back

In [ ]:
# A large network and a small training set: 200 digits for 84,480 weights
X_small, Y_small, y_small = X_tr[:200], Y_tr[:200], y_tr[:200]
big = [64, 256, 256, 10]
print(f'{len(y_small)} digits; a network with {sum(a * b for a, b in zip(big[:-1], big[1:])):,} weights')

def run(label, **kw):
    params, hist = train(init(big, 'he', seed=0), 'relu', 'adam', 0.001, epochs=60, X=X_small, Y=Y_small, **kw)
    print(f'{label:<28} train loss {hist[-1]:.4f}   train acc {accuracy(X_small, y_small, params, "relu"):.3f}   test acc {accuracy(X_te, y_te, params, "relu"):.3f}')
    return params, hist

_, hist_plain = run('no regularisation')
_, hist_decay = run('weight decay 1e-2', decay=1e-2)

In [ ]:
# Early stopping: watch a validation slice, keep the best epoch, give up after ten epochs with no improvement
X_fit, X_val, Y_fit, Y_val = train_test_split(X_small, Y_small, test_size=0.2, random_state=0)
params = init(big, 'he', seed=0)
best_loss, best_epoch, best_params, patience = np.inf, 0, None, 0
for epoch in range(1, 201):
    params, _ = train(params, 'relu', 'adam', 0.001, epochs=1, seed=epoch, X=X_fit, Y=Y_fit)
    val_loss = loss(X_val, Y_val, params, 'relu')
    if val_loss < best_loss:
        best_loss, best_epoch, best_params, patience = val_loss, epoch, [[W.copy(), b.copy()] for W, b in params], 0
    else:
        patience += 1
        if patience == 10:
            break
print(f'{"early stopping":<28} validation loss lowest at epoch {best_epoch}, stopped at {epoch}   test acc {accuracy(X_te, y_te, best_params, "relu"):.3f}')

In [ ]:
# Dropout: during training, silence a random half of each hidden layer
def forward_dropout(X, params, act, rate, rng):
    f, _ = ACT[act]
    activations, masks = [X], []
    for i, (W, b) in enumerate(params):
        z = activations[-1] @ W + b
        if i == len(params) - 1:
            activations.append(softmax(z))
        else:
            a = f(z)
            mask = (rng.uniform(size=a.shape) > rate) / (1 - rate)   # scale up the survivors so the sum is unchanged
            masks.append(mask)
            activations.append(a * mask)
    return activations, masks

def train_dropout(params, act, rate_drop, lr, epochs, X, Y, batch=64, seed=0):
    rng = np.random.default_rng(seed)
    _, df = ACT[act]
    m = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]
    v = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]
    t = 0
    for epoch in range(epochs):
        order = rng.permutation(len(X))
        for start in range(0, len(X), batch):
            idx = order[start:start + batch]
            acts, masks = forward_dropout(X[idx], params, act, rate_drop, rng)
            delta = (acts[-1] - Y[idx]) / len(idx)
            t += 1
            for i in reversed(range(len(params))):
                W, b = params[i]
                gW, gb = acts[i].T @ delta, delta.sum(axis=0)
                if i > 0:
                    delta = (delta @ W.T) * df(acts[i]) * masks[i - 1]
                for j, (P, g) in enumerate([(W, gW), (b, gb)]):
                    m[i][j] = 0.9 * m[i][j] + 0.1 * g
                    v[i][j] = 0.999 * v[i][j] + 0.001 * g ** 2
                    P -= lr * (m[i][j] / (1 - 0.9 ** t)) / (np.sqrt(v[i][j] / (1 - 0.999 ** t)) + 1e-8)
    return params

params = train_dropout(init(big, 'he', seed=0), 'relu', 0.5, 0.001, 60, X_small, Y_small)
print(f'{"dropout 0.5":<28} train acc {accuracy(X_small, y_small, params, "relu"):.3f}   test acc {accuracy(X_te, y_te, params, "relu"):.3f}')

In [ ]:
# And with all 1,257 digits, where the network has enough data to behave
params, hist_all = train(init(big, 'he', seed=0), 'relu', 'adam', 0.001, epochs=60)
print(f'all {len(y_tr)} digits, no regularisation: train acc {accuracy(X_tr, y_tr, params, "relu"):.3f}   test acc {accuracy(X_te, y_te, params, "relu"):.3f}')

## 8. Batch size

In [ ]:
import time
print(f'{"batch":>6} {"updates/epoch":>13} {"loss after 30":>14} {"test acc":>9} {"seconds":>8}')
for batch in [8, 64, 512]:
    start = time.perf_counter()
    params, hist = train(init(sizes, 'he', seed=0), 'relu', 'adam', 0.001, epochs=30, batch=batch)
    print(f'{batch:>6} {int(np.ceil(len(X_tr) / batch)):>13} {hist[-1]:>14.4f} {accuracy(X_te, y_te, params, "relu"):>9.3f} {time.perf_counter() - start:>8.1f}')

## 9. The one line you would ship

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(256, 256), activation='relu', solver='adam',
                                  learning_rate_init=0.001, alpha=1e-3, batch_size=64,
                                  early_stopping=True, validation_fraction=0.2, n_iter_no_change=10,
                                  max_iter=500, random_state=0))
mlp.fit(X_raw_tr, y_tr)
clf = mlp[-1]
print(f'MLPClassifier stopped after {clf.n_iter_} epochs; test accuracy {mlp.score(X_raw_te, y_te):.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_plain, label='200 digits, no regularisation')
ax.plot(hist_decay, label='200 digits, weight decay 1e-2')
ax.plot(hist_all, label='1,257 digits, no regularisation')
ax.set_xlabel('epoch'); ax.set_ylabel('training loss'); ax.set_yscale('log'); ax.legend()
ax.set_title('training loss keeps falling; the test accuracy did not')
plt.show()

## Try it yourself

1. In section 2, make the network ten hidden layers deep. How small do the sigmoid gradients get in the first layer? Does ReLU with He initialisation still hold up?
2. In section 4, give plain SGD a learning rate of 0.5. Does it catch Adam, or diverge?
3. In section 7, try dropout rates of 0.2 and 0.8. Which is better, and what happens to training accuracy at 0.8?